# Topic 19: Generative Models — Exercises

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## Exercise 1: KL Divergence (Hand Calculation & Code)

**Task:** Calculate the KL divergence $D_{\text{KL}}(q \| p)$ where $q = \mathcal{N}(\mu, \Sigma)$ and $p = \mathcal{N}(0, I)$ for a 2D Gaussian.
Given $\mu = [1.0, -1.0]$ and $\Sigma = \text{diag}([e^{0.5}, e^{-0.2}])$ (so log-variance is $[0.5, -0.2]$).

1. Work out the math by hand using the formula $D_{\text{KL}} = -0.5 \sum (1 + \log\sigma^2 - \mu^2 - \sigma^2)$.
2. Verify with code below.

**Solution (Hand Calculation):**
$D_{\text{KL}} = -0.5 [ (1 + 0.5 - 1.0^2 - e^{0.5}) + (1 - 0.2 - (-1.0)^2 - e^{-0.2}) ]$
$e^{0.5} \approx 1.6487, e^{-0.2} \approx 0.8187$
Sum component 1: $1 + 0.5 - 1 - 1.6487 = -1.1487$
Sum component 2: $1 - 0.2 - 1 - 0.8187 = -1.0187$
Total $D_{\text{KL}} = -0.5(-1.1487 - 1.0187) = 1.0837$

In [ ]:
# Implementation
mu = np.array([1.0, -1.0])
logvar = np.array([0.5, -0.2])

def kl_div(mu, logvar):
    return -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar))

ans = kl_div(mu, logvar)
print(f"KL Divergence: {ans:.4f}")
assert np.allclose(ans, 1.0837, atol=1e-3)

## Exercise 2: Reparameterization Trick Gradients

**Task:** Implement the forward pass of $z = \mu + \sigma \odot \epsilon$. Then compute the gradients $\frac{\partial z}{\partial \mu}$ and $\frac{\partial z}{\partial \log\sigma^2}$. Verify these analytic gradients using finite differences.

In [ ]:
mu = np.array([0.5, 0.1])
logvar = np.array([-0.1, 0.2])
eps = np.array([0.4, -0.6])

# 1. Forward
std = np.exp(0.5 * logvar)
z = mu + std * eps

# 2. Analytic gradients
dz_dmu = np.ones_like(mu)
dz_dlogvar = 0.5 * std * eps

# 3. Finite Differences
h = 1e-5
dz_dmu_fd = ((mu + h + std * eps) - z) / h
std_h = np.exp(0.5 * (logvar + h))
dz_dlogvar_fd = ((mu + std_h * eps) - z) / h

print(f"Analytic dz_dlogvar: {dz_dlogvar}")
print(f"FD dz_dlogvar:       {dz_dlogvar_fd}")
assert np.allclose(dz_dmu, dz_dmu_fd, atol=1e-4)
assert np.allclose(dz_dlogvar, dz_dlogvar_fd, atol=1e-4)

## Exercise 3: Conceptual Analysis

**Task:** Explain the fundamental differences between VAE, GAN, and Diffusion in terms of:
1. Training objective
2. Mode coverage (how well they cover the entire data distribution)
3. Sample quality (visual fidelity of outputs)


**Solution:**

1. **Training Objective:**
   - **VAE:** Maximizes the Evidence Lower Bound (ELBO), an explicit approximate likelihood.
   - **GAN:** Solves a minimax game between a Generator and Discriminator (implicit likelihood).
   - **Diffusion:** Matches the score function by learning to denoise at various noise scales (often optimizing a variational bound similar to ELBO).

2. **Mode Coverage:**
   - **VAE:** Excellent mode coverage. The KL divergence penalty encourages spreading mass over the entire data distribution, often leading to blurry outputs.
   - **GAN:** Poor mode coverage. Prone to mode collapse because the generator only needs to produce a few highly realistic modes to fool the discriminator.
   - **Diffusion:** Excellent mode coverage, similar to VAEs but often better due to the flexible diffusion steps.

3. **Sample Quality:**
   - **VAE:** Generally blurry and lower quality due to pixel-wise independent assumptions.
   - **GAN:** Very high fidelity and sharp samples.
   - **Diffusion:** High fidelity, often matching or exceeding GANs, but at the cost of slow iterative sampling.